# Deliverable 2 - INF201 

Submission deadline: 30th Oct 2026

### Hand-in information:
* This is a group submission, so one person submits on behalf of the pair/group
* Submit the Jupyter notebook (ipynb) <i>and</i> a pdf of the same notebooks
    * Should contain code, results/plots and discussion in markdown cells
* Also submit the output files generated by your code
* Include a link to your GitHub repository and a log of the commits
    * To generate the log, paste this into the terminal after your last commit:<br>
    `git log --graph --pretty=format:"%h - %an, %ar : %s" > git_log.txt`
    * The repository should contain a `requirements.txt` with information on your virtual environment
    * Note: Remember to create a new virtual environment for each deliverable
* Include a discussion of GenAI use, for example a summary of the chat log as you have
done earlier
    * This can be a separate file or be included in the Jupyter notebook
* Make a short (5 mins) video discussing your code and results
    * Note that you both have to contribute to the video
    * Either by recording it together
    * Or recording parts separately and editing it together

Gruppemedlemmer: Mahdi Heydari og Noh Yonatan
GitHub: https://github.com/nohjonatans-sketch

In [13]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path("data")

## Scenario  - Campus Coffee Shop Sales Pipeline
You have been hired as a Python developer for the Campus Coffee Shop Chain. The chain operates several coffee stands across different campus buildings. This is an international campus, so the stands accept payment in several currencies.

Each cash register saves transaction logs as text files. However, the legacy system has several issues:
* Mixed Character Encodings: Older registers write text files using ISO-8859-1 (Latin-1) encoding, while newer registers write files using UTF-8.
* System Noise: Registers intersperse valid sales lines with printer and network debug log lines.
* Nested Folder Hierarchy: Log files are stored across nested subdirectories organized by year, store, and register ID.
* Data Errors: System glitches record occasional extreme price outliers (e.g., $999.00), and prices in NOK are by error registered using EUR-scale values (e.g., 4.50 NOK instead of 45.00 NOK).

Your task is to set up Python code that discovers all log files, cleans the transaction data, calculates price metrics and generates summary reports.<br>
Note: In this assignment, you write the functions you will use in tasks 1-4, and then put everything together in task 5. You should therefore read through the entire assignment text before you start, to fully understand what you will use each function for.

### Overview of inputs and outputs for this assignment
Inputs (in `data/` folder)
* You generate the input data on you own laptop by running the script `generate_data_for_d2.py`
* `data/store_config.json`: A nested JSON file containing store metadata.
* `data/2026/...`: A folder hierarchy containing subdirectories with .log files in mixed character encodings.

Outputs (files your code should generate)
* `cleaned_transactions.csv`: A CSV table containing all parsed, outlier-cleaned, and currency-corrected transactions.
* `hourly_revenue.json`: A JSON report containing total revenue per hour per item.
* `status_pivot_table.csv`: A CSV file containing a pivot table of total transaction revenue grouped by item across status categories.
* `item_refund_ratios.csv`: A CSV table containing refund ratios for each item type.

## Task 1: File ingestion and regular expressions

1. Write a robust file reader function:
    * Use `pathlib.Path` to handle file paths.
    * The function should be able to handle both `UTF-8` and `ISO-8859-1/Latin-1` encoding. You may assume that these are the only two encodings the files can have. Note that the function should handle the encoding, you may not hard-code which encoding each file has.
    * Return a tuple containing two strings: `(file_text_content, encoding_used)`.
2. Write a log parser function that uses regular expressions.
    Log files contain valid sales lines mixed with system noise.<br>
    Valid line format:<br>
    `[2026-10-15 08:30:00] [POS_01] item=Latte price=4.50 EUR (SUCCESS)`<br>
    Noise lines to ignore:<br>
    `--- PRINTER PAPER LOW WARNING ---`<br>
    `=== NETWORK SYNC OK ===`<br>
    * Construct a regular expression (`re` module) to capture the 6 key fields from valid lines:
        * `timestamp` (string, format: `YYYY-MM-DD HH:MM:SS`)
        * `register_id` (string, e.g., `POS_01`)
        * `item` (string, e.g., `Latte`)
        * `price` (float, e.g., `4.50`)
        * `currency` (string, e.g., `EUR`, `NOK`, `USD`)
        * `status` (string: `SUCCESS`, `REFUND`, or `CANCELLED`)
    * Return a list of Python dictionaries, where each dictionary represents one valid transaction record.

In [14]:
LINE_PATTERN = re.compile(
    r"^\[(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\] "
    r"\[(?P<register_id>\w+)\] "
    r"item=(?P<item>\w+) "
    r"price=(?P<price>\d+(?:\.\d+)?) "
    r"(?P<currency>[A-Z]{3}) "
    r"\((?P<status>SUCCESS|REFUND|CANCELLED)\)$"
)


def read_log_file(path: Path | str) -> tuple[str, str]:
    """Read a text file, trying UTF-8 first and then Latin-1."""
    raw = Path(path).read_bytes()
    try:
        return raw.decode("utf-8"), "utf-8"
    except UnicodeDecodeError:
        return raw.decode("iso-8859-1"), "iso-8859-1"


def parse_log(text: str) -> list[dict]:
    """Return one dict per valid sales line; noise lines are ignored."""
    records = []
    for line in text.splitlines():
        match = LINE_PATTERN.match(line.strip())
        if match:
            record = match.groupdict()
            record["price"] = float(record["price"])
            records.append(record)
    return records

## Task 2: Recursive utility functions

1. Write a recursive directory search function that finds all log files in a nested directory and returns a list of `pathlib.Path` objects:
    * Convert the input path to a `pathlib.Path` object.
    * Iterate through items using `Path.iterdir()`.
    * If an item is a directory, recursively search inside it.
    * If an item is a file with a matching extension (`.log`), add its Path object to a list.
    * Constraint: Do NOT use `Path.glob()`, `Path.rglob()`, or `os.walk()`. You must implement explicit recursion.
2. Metadata about the different stores can be found in the `data/store_config.json` file. This file contains deeply nested dictionaries and lists (see example below). Write a recursive function that:
    * Traverses any combination of nested dictionaries and lists found in this file.
    * Returns a set (type hint: `set[str]`) of all register id strings where `"active": true`.

Example config.json:
```python
{
  "company": "Campus Coffee",
  "regions": [
    {
      "region_name": "Main Campus",
      "stores": [
        {
          "name": "Library Cafe",
          "registers": [
            {"id": "POS_01", "active": true},
            {"id": "POS_02", "active": false}
          ]
        }
      ]
    }
  ]
}
```

In [15]:
def find_log_files(folder: Path | str) -> list[Path]:
    """Recursively find all .log files below folder."""
    folder = Path(folder)
    log_files = []
    for item in folder.iterdir():
        if item.is_dir():
            log_files.extend(find_log_files(item))
        elif item.is_file() and item.suffix == ".log":
            log_files.append(item)
    return log_files


def find_active_registers(data: dict | list) -> set[str]:
    """Recursively collect the ids of all registers with "active": true."""
    active = set()
    if isinstance(data, dict):
        if data.get("active") is True and "id" in data:
            active.add(data["id"])
        for value in data.values():
            active |= find_active_registers(value)
    elif isinstance(data, list):
        for item in data:
            active |= find_active_registers(item)
    return active

## Task 3: Functions for cleaning and analysing data using NumPy
1. Write a function for removing price outliers that:
    * Converts input price data to a 1D NumPy float array.
    * Detects hardware glitch outliers ($999.00$) where a value deviates from the mean by more than 3 standard deviations:$$\lvert x - \mu \rvert > 3\sigma$$
    * Replaces detected outlier values with the median of the valid prices.
    * Returns the cleaned 1D array.
2. Write a function `compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray` that:
    * Computes the element-wise refund ratio using NumPy array operations:
    $$\text{Refund Ratio} = \frac{\text{Refund Count}}{\text{Sales Count} + 10^{-6}}$$
    * The addition of $\epsilon = 10^{-6}$ prevents zero-division errors.

In [16]:
def remove_outliers(prices) -> np.ndarray:
    """Replace values more than 3 standard deviations from the mean with the median of the valid values."""
    prices = np.asarray(prices, dtype=float).ravel()
    is_outlier = np.abs(prices - prices.mean()) > 3 * prices.std()
    cleaned = prices.copy()
    cleaned[is_outlier] = np.median(prices[~is_outlier])
    return cleaned


def compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray:
    """Refund ratio = refunds / (sales + 1e-6), element-wise."""
    return refund_count / (sales_count + 1e-6)

## Task 4: Data analysis using Pandas
1. Write a function that:
    * Converts the list of transaction dictionaries read from the log files into a `pandas.DataFrame`.
    * Converts the `timestamp` column to `datetime` and sets it as the DataFrame index.
    * Sorts the DataFrame chronologically by timestamp.
2. Write a function that has the DataFrame you just created as input, groups data by item and resamples transactions in 1-hour windows (1h), calculating total price (sum).
3. Write a function that has the DataFrame as input  and creates a pivot table displaying total transaction revenue grouped by item across status categories (`SUCCESS`, `REFUND`, `CANCELLED`).
4. Write a function that has the DataFrame as input and calculates the refund ratio per item by calling the function you made in task 3.

In [17]:
def build_dataframe(records: list[dict]) -> pd.DataFrame:
    """Make a DataFrame with a sorted datetime index from the parsed records."""
    df = pd.DataFrame(records)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    return df.set_index("timestamp").sort_index()


def hourly_revenue(df: pd.DataFrame) -> pd.Series:
    """Total price per item in 1-hour windows."""
    return df.groupby("item")["price"].resample("1h").sum()


def status_pivot(df: pd.DataFrame) -> pd.DataFrame:
    """Total revenue per item for each status."""
    return df.pivot_table(
        values="price", index="item", columns="status", aggfunc="sum", fill_value=0
    )


def item_refund_ratios(df: pd.DataFrame) -> pd.DataFrame:
    """Refund ratio per item (refund count / successful sales count)."""
    counts = df.pivot_table(
        values="price", index="item", columns="status", aggfunc="count", fill_value=0
    )
    ratios = compute_refund_ratio(
        counts["SUCCESS"].to_numpy(), counts["REFUND"].to_numpy()
    )
    return pd.DataFrame({"refund_ratio": ratios}, index=counts.index)

## Task 5: Create full workflow
Combine the functions you have made into a workflow that does the following:
1. Recursively find all log files and extract active register IDs.
2. Ingest text logs and parse records into a list of dictionaries.
3. Build the initial Pandas DataFrame.
4. Apply NumPy outlier cleaning to remove $999.00$ price glitches.
5. Apply currency correction: multiply price by $10.0$ for all transactions where `currency == "NOK"`.
6. Generate resampled reports, pivot tables and refund ratios.
7. Save all output files listed in the introduction (Scenario: Overview of inputs and outputs for this assignment).

In [18]:
# 1. Find log files and active registers
log_files = find_log_files(DATA_DIR)
with open(DATA_DIR / "store_config.json", encoding="utf-8") as f:
    active_registers = find_active_registers(json.load(f))
print(f"Found {len(log_files)} log files, active registers: {sorted(active_registers)}")

# 2. Read and parse the logs (only active registers)
records = []
for log_file in log_files:
    text, encoding = read_log_file(log_file)
    records += [r for r in parse_log(text) if r["register_id"] in active_registers]
print(f"Parsed {len(records)} transactions")

# 3. Build DataFrame
df = build_dataframe(records)

# 4. Remove 999.00 outliers
df["price"] = remove_outliers(df["price"])

# 5. Currency correction: NOK prices were registered in EUR scale
df.loc[df["currency"] == "NOK", "price"] *= 10.0

# 6 + 7. Reports and output files
df.to_csv("cleaned_transactions.csv")

revenue = hourly_revenue(df)
revenue_dict = {
    item: {str(time): round(value, 2) for time, value in group.droplevel(0).items()}
    for item, group in revenue.groupby(level=0)
}
with open("hourly_revenue.json", "w", encoding="utf-8") as f:
    json.dump(revenue_dict, f, indent=2)

pivot = status_pivot(df)
pivot.to_csv("status_pivot_table.csv")

ratios = item_refund_ratios(df)
ratios.to_csv("item_refund_ratios.csv")

print(pivot.round(2))
print(ratios.round(3))

Found 9 log files, active registers: ['POS_01', 'POS_02', 'POS_03']
Parsed 720 transactions
status      CANCELLED  REFUND  SUCCESS
item                                  
Cappuccino     472.92  263.80  1052.88
Croissant      268.48  402.73  1348.47
Espresso       235.19  194.96  1032.21
Latte          240.14  214.19  1443.49
Muffin         203.12  303.41  1162.78
Sandwich      1042.44  962.91  1755.02
            refund_ratio
item                    
Cappuccino         0.253
Croissant          0.282
Espresso           0.259
Latte              0.161
Muffin             0.232
Sandwich           0.296


## Diskusjon

* Filene leses som bytes og dekodes først som UTF-8; hvis det feiler brukes Latin-1. Da trenger vi ikke vite på forhånd hvilken koding hver fil har.
* Regex-en er knyttet til hele linjen (`^...$`), så støylinjer som `[DEBUG] BARCODE SCANNER REBOOTED` ikke blir tatt med selv om de starter med `[`.
* Outlier-fjerningen gjøres *før* NOK-korreksjonen, siden 999.00 ellers kunne blitt ganget med 10.
* Bare registre med `"active": true` i `store_config.json` tas med i analysen.
* Refund ratio regnes som antall refunderinger delt på antall vellykkede salg (`SUCCESS`) per vare.

### Diskusjon

**Oppsummering**

Vi har laget en pipeline som leser loggfiler fra kasseapparatene til Campus Coffee, renser dataene og lager fire rapporter: `cleaned_transactions.csv`, `hourly_revenue.json`, `status_pivot_table.csv` og `item_refund_ratios.csv`. Loggfilene ligger i en nøstet mappestruktur, har ulik tegnkoding og inneholder støylinjer. I tillegg retter løsningen to kjente feil i prisene: feilverdien `999.00` og NOK-priser som er registrert i EUR-skala. Koden er delt i funksjoner (Task 1–4), som settes sammen til én arbeidsflyt i Task 5.

**Valg vi tok**

- **Tegnkoding:** `read_log_file` leser filen som bytes og prøver først UTF-8. Hvis det feiler, brukes Latin-1. Da trenger vi ikke vite på forhånd hvilken koding hver fil har. En Latin-1-fil med for eksempel æ, ø eller å vil gi feil i UTF-8-dekodingen og dermed bli lest som Latin-1. I det utdelte datasettet inneholder Latin-1-filene bare vanlige ASCII-tegn, som er like i begge kodingene, så i praksis blir alle filene lest som UTF-8.

- **Regex:** Mønsteret er forankret i starten og slutten av linjen (`^…$`), så hele linjen må ha riktig format for å bli tatt med. Støylinjer som `[DEBUG] BARCODE SCANNER REBOOTED` begynner med samme tegn som transaksjonslinjene, men blir utelatt fordi resten av linjen ikke passer mønsteret.

- **Rekursjon:** For å finne loggfilene bruker vi `iterdir()` og lar funksjonen kalle seg selv for hver undermappe, uten `glob` eller `os.walk`, slik oppgaven krever. Den samme idéen brukes på JSON-konfigurasjonen, der dictionaries og lister kan ligge inni hverandre i vilkårlig dybde.

- **Avvikende priser:** Priser som ligger mer enn tre standardavvik fra gjennomsnittet, erstattes med medianen av de gyldige prisene. Dette gjøres *før* NOK-korreksjonen. Ellers ville en feilregistrert NOK-pris på `999.00` blitt ganget opp til `9990`.

- **Aktive kasseapparater:** Vi tar bare med transaksjoner fra kasseapparater med `"active": true` i konfigurasjonsfilen. I datasettet gjelder dette POS_01–03, så alle 720 transaksjonene beholdes.

- **Refund ratio:** For hver vare deler vi antall transaksjoner med status `REFUND` på antall med status `SUCCESS`. Vi legger til 10⁻⁶ i nevneren for å unngå deling på null.

**Resultater og tolkning**

Pipelinen fant 9 loggfiler og 720 transaksjoner. I pivot-tabellen har Sandwich høyest sum for vellykkede salg (ca. 1755), etterfulgt av Latte (ca. 1443). Det henger trolig sammen med at Sandwich har høyest pris. Summene består likevel av beløp i NOK, EUR og USD som ikke er regnet om til samme valuta. De kan derfor ikke brukes til å si sikkert hvilken vare som faktisk har høyest omsetning.

Sandwich har også høyest refund ratio (ca. 0,296), etterfulgt av Croissant (ca. 0,282). Latte har lavest (ca. 0,161). Det tilsvarer mellom 16 og 30 refunderinger per 100 vellykkede salg. Generatoren velger imidlertid status tilfeldig, med sannsynlighet 1/6 for `REFUND` og 4/6 for `SUCCESS`, som gir et forventet forhold på 0,25. Forskjellene mellom varene skyldes derfor sannsynligvis tilfeldig variasjon og sier ingenting om kundetilfredshet eller produktkvalitet.

Dataene dekker bare omtrent to timer fra kl. 07.00, så `hourly_revenue.json` har bare noen få tidsvinduer per vare. Det gir lite grunnlag for å se variasjon gjennom dagen.

**Utfordringer**

Det var krevende å lage et regex-mønster som fanget alle gyldige transaksjoner uten å ta med støylinjene. Vi måtte også tenke gjennom rekkefølgen i renseprosessen, spesielt hvorfor avvikende priser må fjernes før NOK-prisene korrigeres. I tillegg fikk vi praktiske problemer: Windows blokkerte en fil i pandas, og datamappen manglet før vi hadde kjørt generatorskriptet. Det viste at både programmiljøet og filstrukturen må være riktig for at koden skal kunne kjøres.

**Begrensninger og mulige forbedringer**

Dataene er syntetiske, så resultatene sier ingenting om det faktiske salget ved Campus Coffee. De viser først og fremst at pipelinen håndterer det utdelte datasettet riktig.

Metoden med tre standardavvik fungerer her fordi `999.00` ligger svært langt unna de andre prisene, som alle er mellom ca. 2,5 og 8 før NOK-korreksjonen. En svakhet er at feilverdiene selv er med i beregningen av gjennomsnitt og standardavvik. Hvis det var mange av dem, ville standardavviket blitt så stort at de kunne unngått å bli oppdaget. En mer robust løsning kunne vært å bruke median og spredning rundt medianen, eller å se etter avvik innenfor hver vare.

NOK-korreksjonen retter bare skalaen på NOK-prisene. Den regner ikke NOK, EUR og USD om til samme valuta. En slik omregning ville vært nødvendig for å sammenligne omsetningen på en meningsfull måte.

### Bruk av generativ KI

Vi brukte KI til å få forklart oppgaveteksten, og hente tips og råd, som vi kunne ta til inspirasjon og læring.

Vi har selv kjørt koden og gått gjennom hver funksjon, slik at vi kan forklare hvordan løsningen fungerer. Vi kontrollerte også resultatene: Feilverdien `999.00` forekommer ikke lenger etter rensingen, og antall transaksjoner stemmer med forventningen (9 filer × 80 linjer = 720). Disse kontrollene gir en viss trygghet for at løsningen fungerer på dette datasettet, men dekker ikke alle mulige feil.